# 08 pamoka – daugiaagentis dizaino šablonas


## Įdiegimas


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Kodėl daugiaagentės sistemos?

Realių pasaulio užduočių, tokių kaip kelionės planavimas, sprendimui reikia įvairių sričių žinių — logistikos, vietinės patirties, biudžeto sudarymo ir kt. Vienas agentas, bandantis viską padaryti, greitai tampa nevaldomas.

Daugiaagentės sistemos tai sprendžia per **specializaciją**: kiekvienas agentas koncentruojasi į vieną sritį, taip gaunant geresnės kokybės rezultatus nei universalistas. Jos taip pat gerina **scalability** — galite pridėti naujų agentų (pvz., skrydžių specialistą, restoranų kritiką) nepriklausomai nuo esamos darbo eigos perrašymo. Agentai susijungia struktūrizuotoje grandinėje, perduodami kontekstą vienas kitam.


## Specializuotų agentų kūrimas


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Darbo eigos sekos kūrimas

`WorkflowBuilder` leidžia sujungti agentus į nukreiptąjį grafiką. Čia sukuriame paprastą dviejų žingsnių seką: **TravelPlanner** sukuria maršrutą, o po to **TravelConcierge** jį peržiūri ir patobulina.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Daugiau agentų pridėjimas į darbo eigą

Viena didžiausių daugiaagentės struktūros privalumų yra tai, kaip lengva ją išplėsti. Žemiau pridedame **BudgetReviewer** agentą, kuris tikrina planą pagal keliautojo biudžetą, pažymi punktus, kurie gali viršyti išlaidų ribą, ir siūlo pinigų taupymo alternatyvas. Darbo eiga dabar vykdo tris agentus paeiliui:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Santrauka

Šioje pamokoje sužinojote, kaip:

1. **Sukurti specializuotus agentus** — kiekvienas su konkrečia funkcija (planavimas, konsjeržas, biudžeto peržiūra).
2. **Sujungti agentus į seką** naudojant `WorkflowBuilder` ir `add_edge`.
3. **Transliuoti išvestį** iš kelių agentų vamzdyno, stebint, kuris agentas kalba.
4. **Išplėsti darbo eigą** pridėjus naujus agentus grandinei nekeičiant esamų.

Daugiagentinis dizaino modelis išlaiko kiekvieną agentą paprastą, o rezultatai yra turtingesni ir išsamiau peržiūrėti nei bet kurio vieno agente būtų įmanoma.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės atsisakymas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, atkreipkite dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas gimtąja kalba turi būti laikomas patikimiausiu šaltiniu. Esant svarbiai informacijai rekomenduojamas profesionalus žmogaus atliekamas vertimas. Mes neatsakome už jokį nesusipratimą ar neteisingą šios vertimo versijos interpretavimą.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
